<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Data to Decisions: GPU-Accelerated Decision Optimization</b></h1>
<h2><b>Exercise 1:</b> Accelerated VRP Solver with cuOpt Python API</h2>

<hr>

## What is the Vehicle Routing Problem (VRP)?

The **Vehicle Routing Problem** is one of the most important optimization problems in logistics. Given:
- A set of **customers** with locations and demands
- A fleet of **vehicles** with limited capacity
- A **depot** where vehicles start and end

**Goal**: Find the optimal set of routes that minimizes total travel cost while serving all customers.

### CVRP (Capacitated VRP)
The most common variant where each vehicle has a **capacity limit** - it can only carry so much before returning to the depot.

**Real-world examples**: Package delivery (UPS, FedEx), food delivery, field service routing, school bus planning.

There are many other variants of VRP to account for a variety of business rules: VRP with time windows, (paired) pickup and deliveries, prize collection, driver breaks, ...

### Why is VRP Hard?
VRP is **[NP-hard](https://en.wikipedia.org/wiki/NP-hardness)**, meaning finding the guaranteed optimal solution becomes computationally intractable as the problem grows. With just 20 customers, there are more possible route combinations than atoms in the universe! In practice, we use **heuristics** and **metaheuristics** to find high-quality solutions in reasonable time. cuOpt implements these heuristics on GPUs, achieving orders of magnitude speedup over traditional approaches.

---

**What you'll learn:**
1. Solve a basic **CVRP** - assign deliveries to vehicles with capacity limits
2. Add **Time Windows** - customers must be visited within specific time slots

**cuOpt** uses GPU acceleration to solve these problems orders of magnitude faster than traditional solvers running on CPU.


In [ ]:
# Setup - run this cell first
# Uncomment the pip install line if running in Google Colab
# %pip install --extra-index-url=https://pypi.nvidia.com cuopt-cu13

import numpy as np
import pandas as pd
import cudf
from cuopt import routing
import matplotlib.pyplot as plt

print("✅ cuOpt ready!")


---
## Part 1: Basic CVRP (Capacitated Vehicle Routing)

**Scenario**: A warehouse needs to deliver goods to 7 customers using 3 vehicles. Each vehicle has limited capacity.

**Goal**: Minimize total distance while ensuring all demands are met.

Let's start by defining ingredients of the VRP problem, namely locations to be visited (including depot), quantities demanded by each location and capacities of the vehicles.


In [ ]:
# Define the problem
location_names = ["Depot", "A", "B", "C", "D", "E", "F", "G"]
coords = [[4,4], [1,3], [8,1], [2,1], [6,7], [0,2], [7,6], [5,3]]

# Demand at each location (depot = 0)
demand = [0, 4, 4, 2, 2, 1, 2, 1]  # Total: 16 units

# Vehicle capacities
capacities = [8, 8, 8]  # Total: 24 units (enough for all demand)

print(f"Locations: {len(location_names)}, Vehicles: {len(capacities)}")
print(f"Total demand: {sum(demand)}, Total capacity: {sum(capacities)}")

Next we will define a function that calculates Euclidean (straight-line) distances betweene locations:

In [ ]:
# Helper: compute distance matrix from coordinates
def compute_distance_matrix(coords):
    """Compute Euclidean distance matrix from [x, y] coordinates."""
    n = len(coords)
    matrix = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(n):
            dx = coords[i][0] - coords[j][0]
            dy = coords[i][1] - coords[j][1]
            matrix[i, j] = np.sqrt(dx*dx + dy*dy)
    return matrix

Now we can build the cuOpt data model with the information specified above.

In [ ]:
# Build the cuOpt model
n_locations = len(location_names)
n_vehicles = len(capacities)

# Create data model
model = routing.DataModel(n_locations, n_vehicles)

# Compute distance matrix from coordinates
distance_matrix = compute_distance_matrix(coords)

# Add cost matrix (what we want to minimize)
model.add_cost_matrix(cudf.DataFrame(distance_matrix))

# Add capacity constraint
model.add_capacity_dimension(
    "demand",
    cudf.Series(demand, dtype=np.int32),
    cudf.Series(capacities, dtype=np.int32)
)

# All vehicles start and end at depot (index 0)
model.set_vehicle_locations(
    cudf.Series([0]*n_vehicles, dtype=np.int32),
    cudf.Series([0]*n_vehicles, dtype=np.int32)
)

print("✅ Model built!")


The next step is to solve the VRP problem by invoking the ```Solve``` function. We also set a time limit for the solver of 2 seconds.

Once the problem is solved, we output solution information about the total distance traveled, number of vehicles used and the individual routes created.

In [ ]:
# Specify time limit for the solver
settings = routing.SolverSettings()
settings.set_time_limit(2)

# Find the solution
solution = routing.Solve(model, settings)

if solution.get_status() == 0:
    print(f"✅ Solution found!")
    print(f"   Total distance: {solution.get_total_objective():.1f}")
    print(f"   Vehicles used: {solution.get_vehicle_count()}")
    
    # Show routes
    routes = solution.get_route().to_pandas()
    for v in routes['truck_id'].unique():
        route = routes[routes['truck_id']==v]['route'].tolist()
        path = ' → '.join([location_names[i] for i in route])
        print(f"   Vehicle {v}: {path}")

Next we will visualize the routes with the help of a ```plot_routes``` helper function.

In [ ]:
# Helper function to visualize routes
def plot_routes(coords, solution, location_names, title="Routes"):
    colors = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6"]
    plt.figure(figsize=(10, 8))
    
    # Plot depot and locations
    plt.scatter(coords[0][0], coords[0][1], c='green', s=200, marker='s', label='Depot', zorder=5)
    for i in range(1, len(coords)):
        plt.scatter(coords[i][0], coords[i][1], c='red', s=150, zorder=5)
    
    # Draw routes
    routes_df = solution.get_route().to_pandas()
    for v_id in routes_df['truck_id'].unique():
        route = routes_df[routes_df['truck_id'] == v_id]['route'].tolist()
        color = colors[v_id % len(colors)]
        for i in range(len(route)-1):
            arrow_props = dict(arrowstyle="->", edgecolor=color, lw=2, mutation_scale=40)
            plt.annotate("", xy=coords[route[i+1]], xytext=coords[route[i]], arrowprops=arrow_props)
        plt.plot([], [], color=color, lw=2, label=f'Vehicle {v_id}')

    for i in range(1, len(coords)):
        plt.annotate(location_names[i], coords[i], fontsize=11, xytext=(5,5), textcoords='offset points')
    
    plt.title(title, fontsize=14)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# Visualize
plot_routes(coords, solution, location_names, "CVRP Solution")

How do the routes look?

---
## Part 2: Adding Time Windows (CVRPTW)

Next, we will model a more realistic representation of an actual delivery optimization problem.

**New constraint**: Each customer can only be visited within a specific time window.

This is critical for real-world scenarios:
- 🍔 Restaurant deliveries during lunch hours
- 🏠 Home deliveries when customers are available
- 🏢 Business deliveries during office hours

First, let's specify our desired time specifications. Below, we will define:
- A time window to indicate the operating time for each location.
- Service times for a typical delivery in each location. 

In [ ]:
# Time windows [earliest, latest] in minutes from 8:00 AM
time_windows = {
    ## Let's assume the depot is open for a whole day (8 hours, or 480 mins)
    "Depot": [0, 480],     # Open all day
    ## We can then specify the various stores within the depot
    "A": [30, 90],         # 8:30 - 9:30 AM
    "B": [120, 180],       # 10:00 - 11:00 AM  
    "C": [60, 150],        # 9:00 - 10:30 AM
    "D": [180, 300],       # 11:00 AM - 1:00 PM
    "E": [0, 120],         # 8:00 - 10:00 AM
    "F": [240, 360],       # 12:00 - 2:00 PM
    "G": [60, 240],        # 9:00 AM - 12:00 PM
}

# Service time at each stop (minutes)
service_times = [0, 10, 15, 10, 10, 5, 15, 10]

# Display time windows
print("Time Windows (minutes from 8:00 AM):")
for loc, (early, late) in time_windows.items():
    print(f"  {loc}: {early:3d} - {late:3d} min")

We next build the cuOpt data model similar to above, and this time we will add 3 new things:
- travel time matrix (so that we can keep track of meeting time windows accurately
- time windows for each location
- service times at each location

In [ ]:
# Build model with time windows
model_tw = routing.DataModel(n_locations, n_vehicles)

# Same as before: cost matrix, capacity, vehicle locations
model_tw.add_cost_matrix(cudf.DataFrame(distance_matrix))
model_tw.add_capacity_dimension("demand", 
    cudf.Series(demand, dtype=np.int32),
    cudf.Series(capacities, dtype=np.int32))
model_tw.set_vehicle_locations(
    cudf.Series([0]*n_vehicles, dtype=np.int32),
    cudf.Series([0]*n_vehicles, dtype=np.int32))

# NEW: Add travel time matrix (10 min per distance unit)
travel_time = (distance_matrix * 10).astype(np.float32)
model_tw.add_transit_time_matrix(cudf.DataFrame(travel_time))

# NEW: Add time windows
earliest = cudf.Series([tw[0] for tw in time_windows.values()], dtype=np.int32)
latest = cudf.Series([tw[1] for tw in time_windows.values()], dtype=np.int32)
model_tw.set_order_time_windows(earliest, latest)

# NEW: Add service times
model_tw.set_order_service_times(cudf.Series(service_times, dtype=np.int32))

print("✅ Model with time windows built!")


Now we can solve the problem and display the solution as before:

In [ ]:
# Solve with time windows
solution_tw = routing.Solve(model_tw, settings)

if solution_tw.get_status() == 0:
    print(f"✅ Solution with time windows found!")
    print(f"   Total distance: {solution_tw.get_total_objective():.1f}")
    
    routes = solution_tw.get_route().to_pandas()
    for v in routes['truck_id'].unique():
        route = routes[routes['truck_id']==v]['route'].tolist()
        path = ' → '.join([location_names[i] for i in route])
        print(f"   Vehicle {v}: {path}")
else:
    print("❌ No feasible solution - time windows may be too tight!")


In [ ]:
# Visualize time window solution
# plot_routes(coords, solution, location_names, "CVRP Solution")  ## Previous solution
plot_routes(coords, solution_tw, location_names, "CVRPTW Solution (with Time Windows)")

---
## Bonus: Gehring & Homberger Benchmark (200 Customers)

Let's test cuOpt on a **real benchmark** - the [Gehring & Homberger VRPTW benchmark](https://www.sintef.no/projectweb/top/vrptw/200-customers/), an extended version of Solomon's classic benchmark with larger instances.

**Instance R1_2_3**: 200 customers with randomly distributed locations.
- Best known solution: **18 vehicles, 3381.96 distance**

Can cuOpt find a competitive solution in just a few seconds?


In [ ]:
# Gehring & Homberger R1_2_3 benchmark - 200 customers (randomly distributed)
# Source: https://www.sintef.no/projectweb/top/vrptw/200-customers/
# Format: [x, y, demand, earliest, latest, service_time]
# Vehicle capacity: 200, Time horizon: 0-634

r1_2_3_data = [
    [70, 70, 0, 0, 634, 0],       # Depot
    [107, 77, 34, 0, 587, 10], [109, 139, 8, 123, 133, 10], [120, 22, 39, 124, 134, 10],
    [48, 47, 19, 31, 41, 10], [116, 22, 32, 0, 558, 10], [12, 138, 14, 285, 295, 10],
    [86, 40, 22, 0, 590, 10], [121, 124, 21, 0, 550, 10], [61, 57, 35, 385, 395, 10],
    [40, 113, 23, 487, 497, 10], [129, 24, 18, 0, 550, 10], [12, 84, 16, 209, 219, 10],
    [44, 116, 27, 0, 572, 10], [102, 52, 15, 0, 588, 10], [41, 36, 13, 61, 71, 10],
    [132, 133, 24, 0, 536, 10], [104, 139, 29, 0, 548, 10], [104, 54, 13, 0, 587, 10],
    [22, 104, 11, 0, 566, 10], [46, 133, 7, 86, 96, 10], [138, 78, 43, 212, 222, 10],
    [16, 92, 10, 0, 566, 10], [18, 104, 18, 347, 357, 10], [66, 82, 27, 0, 612, 10],
    [107, 25, 13, 284, 294, 10], [139, 73, 10, 97, 107, 10], [101, 0, 9, 0, 548, 10],
    [90, 14, 31, 0, 565, 10], [20, 69, 13, 388, 398, 10], [64, 132, 8, 67, 77, 10],
    [115, 82, 30, 0, 578, 10], [54, 106, 9, 0, 585, 10], [30, 21, 12, 0, 561, 10],
    [63, 129, 7, 216, 226, 10], [82, 100, 18, 114, 124, 10], [108, 30, 18, 106, 116, 10],
    [37, 73, 33, 0, 591, 10], [50, 112, 18, 242, 252, 10], [47, 83, 1, 328, 338, 10],
    [92, 138, 12, 412, 422, 10], [81, 36, 17, 0, 589, 10], [115, 124, 11, 421, 431, 10],
    [13, 48, 15, 0, 563, 10], [113, 35, 9, 60, 70, 10], [60, 84, 23, 0, 607, 10],
    [44, 58, 30, 28, 38, 10], [8, 87, 17, 0, 560, 10], [116, 105, 23, 154, 164, 10],
    [117, 4, 9, 0, 543, 10], [140, 42, 13, 0, 549, 10], [139, 68, 25, 88, 98, 10],
    [58, 95, 35, 495, 505, 10], [103, 119, 18, 0, 565, 10], [32, 52, 32, 161, 171, 10],
    [89, 39, 23, 36, 46, 10], [54, 41, 9, 0, 591, 10], [39, 100, 31, 76, 86, 10],
    [5, 99, 24, 0, 553, 10], [67, 96, 26, 0, 598, 10], [73, 96, 9, 0, 598, 10],
    [112, 13, 27, 0, 554, 10], [55, 127, 17, 0, 566, 10], [103, 44, 12, 42, 52, 10],
    [97, 12, 10, 0, 561, 10], [16, 62, 15, 0, 570, 10], [140, 36, 1, 0, 547, 10],
    [90, 44, 6, 62, 72, 10], [113, 32, 36, 154, 164, 10], [104, 73, 17, 259, 269, 10],
    [36, 38, 5, 0, 578, 10], [15, 4, 23, 111, 121, 10], [54, 33, 20, 0, 584, 10],
    [70, 94, 32, 24, 34, 10], [35, 90, 20, 456, 466, 10], [88, 29, 33, 570, 580, 10],
    [109, 131, 16, 236, 246, 10], [10, 111, 21, 0, 552, 10], [138, 105, 24, 413, 423, 10],
    [113, 45, 4, 309, 319, 10], [92, 22, 18, 0, 572, 10], [85, 75, 37, 0, 609, 10],
    [10, 86, 14, 0, 562, 10], [51, 92, 9, 0, 595, 10], [46, 112, 10, 296, 306, 10],
    [12, 48, 4, 0, 562, 10], [137, 7, 40, 166, 176, 10], [79, 80, 4, 13, 23, 10],
    [69, 116, 26, 354, 364, 10], [28, 62, 23, 0, 582, 10], [50, 106, 25, 0, 583, 10],
    [83, 38, 22, 0, 590, 10], [120, 102, 18, 107, 117, 10], [65, 0, 20, 84, 94, 10],
    [4, 71, 19, 170, 180, 10], [19, 120, 16, 271, 281, 10], [18, 70, 26, 401, 411, 10],
    [58, 80, 18, 0, 609, 10], [49, 140, 3, 0, 551, 10], [69, 115, 20, 0, 579, 10],
    [36, 124, 34, 0, 561, 10], [84, 88, 16, 366, 376, 10], [17, 138, 8, 443, 453, 10],
    [103, 82, 27, 0, 589, 10], [38, 122, 13, 244, 254, 10], [70, 83, 23, 495, 505, 10],
    [29, 60, 14, 42, 52, 10], [61, 34, 7, 0, 587, 10], [54, 78, 11, 560, 570, 10],
    [83, 12, 29, 240, 250, 10], [34, 72, 21, 0, 588, 10], [80, 103, 15, 382, 392, 10],
    [78, 111, 22, 0, 583, 10], [116, 83, 30, 0, 577, 10], [29, 94, 1, 0, 577, 10],
    [137, 137, 27, 110, 120, 10], [110, 28, 20, 315, 325, 10], [116, 13, 5, 0, 551, 10],
    [59, 115, 26, 324, 334, 10], [71, 70, 9, 490, 500, 10], [120, 60, 18, 64, 74, 10],
    [85, 111, 25, 0, 581, 10], [121, 123, 9, 0, 551, 10], [137, 84, 41, 0, 556, 10],
    [103, 86, 19, 268, 278, 10], [30, 57, 20, 240, 250, 10], [139, 72, 23, 0, 555, 10],
    [2, 106, 16, 0, 548, 10], [64, 138, 15, 0, 556, 10], [23, 39, 20, 0, 568, 10],
    [126, 23, 13, 0, 551, 10], [138, 24, 12, 0, 542, 10], [78, 63, 19, 10, 20, 10],
    [98, 4, 20, 0, 553, 10], [103, 77, 17, 301, 311, 10], [58, 11, 9, 0, 564, 10],
    [26, 130, 20, 278, 288, 10], [104, 32, 18, 145, 155, 10], [55, 102, 4, 325, 335, 10],
    [137, 101, 12, 0, 551, 10], [65, 80, 20, 11, 21, 10], [26, 50, 7, 0, 576, 10],
    [56, 139, 18, 320, 330, 10], [119, 73, 15, 127, 137, 10], [9, 19, 10, 383, 393, 10],
    [82, 103, 15, 45, 55, 10], [107, 102, 12, 0, 576, 10], [60, 23, 16, 0, 576, 10],
    [123, 84, 13, 426, 436, 10], [79, 72, 22, 0, 615, 10], [99, 34, 9, 0, 578, 10],
    [27, 46, 35, 219, 229, 10], [64, 115, 15, 0, 579, 10], [32, 6, 11, 105, 115, 10],
    [86, 80, 21, 0, 606, 10], [81, 57, 8, 17, 27, 10], [135, 65, 3, 0, 559, 10],
    [66, 40, 20, 30, 40, 10], [120, 47, 18, 0, 569, 10], [37, 125, 10, 353, 363, 10],
    [120, 14, 20, 0, 549, 10], [114, 109, 16, 0, 566, 10], [137, 127, 28, 206, 216, 10],
    [37, 138, 5, 220, 230, 10], [36, 54, 15, 0, 587, 10], [109, 78, 21, 248, 258, 10],
    [34, 20, 15, 161, 171, 10], [39, 13, 20, 0, 560, 10], [137, 96, 9, 0, 553, 10],
    [110, 99, 21, 313, 323, 10], [136, 123, 31, 329, 339, 10], [133, 130, 13, 232, 242, 10],
    [120, 52, 20, 384, 394, 10], [33, 117, 20, 0, 565, 10], [37, 79, 5, 0, 590, 10],
    [60, 54, 3, 0, 606, 10], [125, 68, 20, 202, 212, 10], [13, 129, 25, 361, 371, 10],
    [20, 106, 24, 0, 563, 10], [15, 9, 2, 420, 430, 10], [106, 10, 13, 0, 555, 10],
    [100, 99, 17, 309, 319, 10], [89, 24, 15, 311, 321, 10], [67, 120, 5, 0, 574, 10],
    [42, 29, 12, 70, 80, 10], [117, 92, 25, 0, 573, 10], [15, 18, 10, 0, 549, 10],
    [139, 110, 19, 0, 545, 10], [44, 36, 9, 0, 582, 10], [8, 138, 10, 0, 532, 10],
    [33, 0, 20, 492, 502, 10], [4, 44, 24, 0, 554, 10], [97, 115, 25, 0, 572, 10],
    [23, 19, 16, 0, 555, 10], [56, 129, 21, 0, 564, 10], [105, 2, 1, 0, 548, 10],
    [66, 91, 16, 21, 31, 10], [93, 61, 10, 531, 541, 10], [27, 104, 1, 0, 570, 10],
    [69, 6, 8, 93, 103, 10], [95, 135, 27, 411, 421, 10],
]

# Extract data
coords_bench = [[d[0], d[1]] for d in r1_2_3_data]
demand_bench = [d[2] for d in r1_2_3_data]
earliest_bench = [d[3] for d in r1_2_3_data]
latest_bench = [d[4] for d in r1_2_3_data]
service_bench = [d[5] for d in r1_2_3_data]

print(f"Gehring & Homberger R1_2_3: {len(r1_2_3_data)-1} customers")
print(f"Total demand: {sum(demand_bench)}")


In [ ]:
# Solve Gehring & Homberger R1_2_3 benchmark
n_bench = len(r1_2_3_data)
n_vehicles_bench = 50  # Provide enough vehicles

# Build model
model_bench = routing.DataModel(n_bench, n_vehicles_bench)

# Distance/cost matrix
dist_bench = compute_distance_matrix(coords_bench)
model_bench.add_cost_matrix(cudf.DataFrame(dist_bench))

# Travel time (1 time unit per distance unit for this benchmark)
model_bench.add_transit_time_matrix(cudf.DataFrame(dist_bench.astype(np.float32)))

# Capacity (Gehring & Homberger uses 200)
model_bench.add_capacity_dimension("demand",
    cudf.Series(demand_bench, dtype=np.int32),
    cudf.Series([200]*n_vehicles_bench, dtype=np.int32))

# Time windows
model_bench.set_order_time_windows(
    cudf.Series(earliest_bench, dtype=np.int32),
    cudf.Series(latest_bench, dtype=np.int32))
model_bench.set_order_service_times(cudf.Series(service_bench, dtype=np.int32))

# Vehicle locations
model_bench.set_vehicle_locations(
    cudf.Series([0]*n_vehicles_bench, dtype=np.int32),
    cudf.Series([0]*n_vehicles_bench, dtype=np.int32))

In [ ]:
# Solve with 1 min time limit, increase it to improve the solution
settings_bench = routing.SolverSettings()
settings_bench.set_time_limit(60)

solution_bench = routing.Solve(model_bench, settings_bench)

if solution_bench.get_status() == 0:
    print(f"✅ R1_2_3 solved!")
    print(f"   Vehicles used: {solution_bench.get_vehicle_count()}")
    print(f"   Total distance: {solution_bench.get_total_objective():.2f}")
    print(f"\n   Best known: 18 vehicles, 3381.96 distance")
else:
    print("❌ No feasible solution found")

---
## Summary

<div style="float: left">
    
| Step | Method | Purpose |
|------|--------|---------|
| 1 | `DataModel(n_loc, n_veh)` | Create model |
| 2 | `add_cost_matrix()` | Distance/cost to minimize |
| 3 | `add_capacity_dimension()` | Vehicle capacity limits |
| 4 | `set_vehicle_locations()` | Depot locations |
| 5 | `add_transit_time_matrix()` | Travel times |
| 6 | `set_order_time_windows()` | Customer availability |
| 7 | `Solve(model, settings)` | Find optimal routes |

</div>

**Other cuOpt features**: [Pickup & delivery pairs](https://docs.nvidia.com/cuopt/user-guide/latest/routing-features.html#pickup-and-deliveries), [heterogeneous fleets](https://docs.nvidia.com/cuopt/user-guide/latest/routing-features.html#heterogeneous-fleet), [breaks](https://docs.nvidia.com/cuopt/user-guide/latest/routing-features.html#vehicle-breaks), and [more](https://docs.nvidia.com/cuopt/user-guide/latest/routing-features.html#routing-features)!

📚 [Full documentation](https://docs.nvidia.com/cuopt/)


**Congratulations!** You finished this exercise by modeling two variants of a small VRP problem with cuOpt APIand solving it.

**In the next exercise, you will work with a Mixed-Integer Linear Programming model and solve it in the context of supply chain optimization.**

<img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>